# 02 — Audio Preprocessing

Converts raw `.mp3` recordings (`data/raw/`) into **mel-spectrogram PNG images** (`data/processed/`).

| Step | Description |
|------|-------------|
| 1. | Setup |
| 2. | Configure parameters |
| 3. | Preview waveform + spectrogram |
| 4. | Visualise spectrograms across species |
| 5. | Run full pipeline |
| 6. | Dataset summary |

**Local (recommended):** run from the repo root with `jupyter notebook notebooks/02_preprocessing.ipynb`.  
**Google Colab:** run the setup cell below — data downloaded by notebook 01 must be present in the session.

## 1. Setup — Colab / local environment

In [ ]:
import sys, os

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')

    repo = '/content/bird-acoustics-classifier'
    if not os.path.exists(repo):
        !git clone -q https://github.com/danort92/bird-acoustics-classifier.git {repo}
    %cd /content/bird-acoustics-classifier
    !pip install -q librosa soundfile tqdm

    # Link data dirs to Drive so data persists across sessions
    for subdir in ['data/raw', 'data/processed']:
        drive_path = f'/content/drive/MyDrive/bird-acoustics-classifier/{subdir}'
        os.makedirs(drive_path, exist_ok=True)
        if os.path.islink(subdir) or os.path.isdir(subdir):
            !rm -rf {subdir}
        !ln -s {drive_path} {subdir}

    sys.path.insert(0, repo)
    print('Colab ready. Using data from Google Drive.')
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')
    sys.path.insert(0, os.getcwd())
    print('Working directory:', os.getcwd())


In [ ]:
import sys
sys.path.insert(0, '.')

import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import librosa
import librosa.display

from src.preprocessing import SpectrogramConverter, AudioConfig

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams['figure.dpi'] = 110

print('librosa version:', librosa.__version__)

## 2. Configuration

Adjust these values to experiment with different spectrogram representations.

In [ ]:
# ─── Audio / spectrogram parameters ──────────────────────────────────────────
SAMPLE_RATE   = 22050      # Hz — standard for bioacoustics
CLIP_DURATION = 5.0        # seconds per image tile
CLIP_OVERLAP  = 0.0        # fraction of overlap between tiles (0 = no overlap)
N_MELS        = 128        # mel filter banks  (image height)
N_FFT         = 2048       # FFT window size
HOP_LENGTH    = 512        # hop between frames
F_MIN         = 500.0      # low-frequency cut-off (Hz)  — reduces wind noise
F_MAX         = 15000.0    # high-frequency cut-off (Hz)
TOP_DB        = 80.0       # dynamic range (dB)
IMG_SIZE      = (224, 224) # output image size in pixels (W × H)

# ─── Paths — plain relative paths, identical in Colab and locally ─────────────
RAW_DIR       = 'data/raw'
PROCESSED_DIR = 'data/processed'

# ─── Build config & converter ─────────────────────────────────────────────────
config = AudioConfig(
    sample_rate=SAMPLE_RATE,
    clip_duration=CLIP_DURATION,
    clip_overlap=CLIP_OVERLAP,
    n_mels=N_MELS,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    f_min=F_MIN,
    f_max=F_MAX,
    top_db=TOP_DB,
    img_size=IMG_SIZE,
)

converter = SpectrogramConverter(output_dir=PROCESSED_DIR, config=config)

print('AudioConfig:')
for k, v in config.__dict__.items():
    print(f'  {k:20s} = {v}')

## 3. Preview — waveform + spectrogram for one recording

Picks the first `.mp3` available in `data/raw/` and displays clip 0.

In [ ]:
raw_path = Path(RAW_DIR)
mp3_files = sorted(raw_path.rglob('*.mp3'))

if not mp3_files:
    print('⚠  No .mp3 files found in', RAW_DIR)
    print('   Run 01_download.ipynb first.')
else:
    sample_mp3 = mp3_files[0]
    print(f'Sample file: {sample_mp3}')
    fig = converter.plot_waveform_and_spectrogram(sample_mp3, clip_index=0)
    plt.show()

### 3b. Multiple clips from the same recording

Each long recording is sliced into non-overlapping `CLIP_DURATION`-second windows.  
Here we visualise the first 4 clips of the same file.

In [ ]:
if mp3_files:
    y, sr = librosa.load(sample_mp3, sr=SAMPLE_RATE, mono=True)
    clip_samples = int(CLIP_DURATION * SAMPLE_RATE)
    n_clips = min(4, len(y) // clip_samples)

    if n_clips == 0:
        print('Recording is shorter than one clip — only clip 0 available.')
        n_clips = 1

    fig, axes = plt.subplots(1, n_clips, figsize=(4 * n_clips, 3))
    if n_clips == 1:
        axes = [axes]

    for i, ax in enumerate(axes):
        try:
            converter.plot_spectrogram(sample_mp3, clip_index=i, ax=ax,
                                       title=f'Clip {i}')
        except IndexError:
            ax.set_visible(False)

    fig.suptitle(sample_mp3.name, fontsize=10)
    fig.tight_layout()
    plt.show()

## 4. Species grid — one spectrogram per species

Selects the first available recording for each species and shows clip 0.  
This gives a quick visual sanity-check of the variety in the dataset.

In [ ]:
species_dirs = sorted([d for d in raw_path.iterdir() if d.is_dir()])

if not species_dirs:
    print('⚠  No species directories found in', RAW_DIR)
else:
    # Collect one representative mp3 per species
    representatives = []
    for sd in species_dirs:
        mp3s = sorted(sd.glob('*.mp3'))
        if mp3s:
            representatives.append((sd.name, mp3s[0]))

    n = len(representatives)
    if n == 0:
        print('No .mp3 files found inside species directories.')
    else:
        cols = min(4, n)
        rows = (n + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 3.5 * rows))
        axes_flat = np.array(axes).flatten()

        for ax in axes_flat:
            ax.set_visible(False)

        for ax, (species, mp3) in zip(axes_flat, representatives):
            ax.set_visible(True)
            try:
                converter.plot_spectrogram(mp3, clip_index=0, ax=ax,
                                           title=species.replace('_', ' '))
            except Exception as e:
                ax.set_title(f'{species}\n(error: {e})')

        fig.suptitle('Mel spectrograms — one per species', fontsize=13, y=1.01)
        fig.tight_layout()
        plt.show()
        print(f'Displayed {len(representatives)} / {len(species_dirs)} species')

## 4b. Effect of N_MELS on spectrogram resolution

Shows the same clip converted with 64, 128, and 256 mel bins side-by-side.

In [ ]:
if mp3_files:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, n_mels_val in zip(axes, [64, 128, 256]):
        cfg_tmp = AudioConfig(
            sample_rate=SAMPLE_RATE, clip_duration=CLIP_DURATION,
            n_mels=n_mels_val, n_fft=N_FFT, hop_length=HOP_LENGTH,
            f_min=F_MIN, f_max=F_MAX, top_db=TOP_DB,
        )
        conv_tmp = SpectrogramConverter(config=cfg_tmp)
        conv_tmp.plot_spectrogram(sample_mp3, ax=ax, title=f'n_mels={n_mels_val}')
    fig.suptitle('Effect of N_MELS on frequency resolution', fontsize=12)
    fig.tight_layout()
    plt.show()

## 5. Run full preprocessing

Processes **all** species in `data/raw/` and writes PNG files to `data/processed/`.  
Long recordings are automatically sliced into `CLIP_DURATION`-second clips.  
Set `overwrite=True` to regenerate images even if they already exist.

In [ ]:
OVERWRITE = False   # set True to force re-generation

summary = converter.process_all(input_dir=RAW_DIR, overwrite=OVERWRITE)

print('\nClips per species:')
for species, n_clips in sorted(summary.items()):
    print(f'  {species:<40} {n_clips:>4} clips')

## 6. Dataset summary

In [ ]:
processed_path = Path(PROCESSED_DIR)
species_counts = {}

for sp_dir in sorted(processed_path.iterdir()):
    if sp_dir.is_dir():
        count = len(list(sp_dir.glob('*.png')))
        if count > 0:
            species_counts[sp_dir.name] = count

total = sum(species_counts.values())

print(f'Total species  : {len(species_counts)}')
print(f'Total clips    : {total}')
print(f'Image size     : {IMG_SIZE[0]}×{IMG_SIZE[1]} px')
print(f'Clip duration  : {CLIP_DURATION}s @ {SAMPLE_RATE} Hz')

# Bar chart
if species_counts:
    fig, ax = plt.subplots(figsize=(max(8, len(species_counts) * 0.6), 5))
    names  = [n.replace('_', '\n') for n in species_counts]
    counts = list(species_counts.values())
    bars = ax.bar(range(len(names)), counts, color='steelblue', edgecolor='white')
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, fontsize=7)
    ax.set_ylabel('Number of clips')
    ax.set_title(f'Clip distribution across {len(species_counts)} species  (total = {total})')
    ax.bar_label(bars, fontsize=7, padding=2)
    fig.tight_layout()
    plt.show()

In [ ]:
# Show a random sample of saved PNG images from data/processed/
import random
from PIL import Image   # Pillow is pre-installed on Colab & most envs

png_files = list(processed_path.rglob('*.png'))

if not png_files:
    print('No PNG files found — run cell 5 first.')
else:
    sample = random.sample(png_files, min(12, len(png_files)))
    cols = 4
    rows = (len(sample) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    for ax in np.array(axes).flatten():
        ax.set_visible(False)
    for ax, p in zip(np.array(axes).flatten(), sample):
        ax.set_visible(True)
        img = Image.open(p)
        ax.imshow(img)
        ax.set_title(p.parent.name.replace('_', ' '), fontsize=7)
        ax.axis('off')
    fig.suptitle('Random sample of processed spectrogram images', fontsize=11)
    fig.tight_layout()
    plt.show()

---
## CLI alternative

The full preprocessing pipeline can also be run from the terminal using `scripts/preprocess.py`, which reads audio parameters from `config/default.yaml`:

```bash
# Use defaults from config/default.yaml
python scripts/preprocess.py

# Force overwrite of existing images
python scripts/preprocess.py --overwrite

# Custom input/output directories
python scripts/preprocess.py --input-dir data/raw --output-dir data/processed

# All options
python scripts/preprocess.py --help
```

Edit `config/default.yaml` to change `sample_rate`, `clip_duration`, `n_mels`, and other parameters without modifying the code.

---
**Next step:** `03_training.ipynb` — train EfficientNet on the spectrogram dataset *(coming)*